# Predictive Maintenance: Degradation Forecasting & Anomaly Detection

**Read this first**: every dataset in this notebook is **synthetic**. The hydraulic baseline (head loss at a given roughness) is computed via this project's real, verified Darcy-Weisbach engine — but the degradation trend over time, sensor noise, and injected anomalies are all fabricated for demonstration. This notebook shows the ML *pattern* — how you'd structure training and inference for pipe predictive maintenance — not a validated predictive tool. Swap in your own real inspection records / sensor logs for actual use.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

## Part 1: Roughness Degradation Forecasting

### Generate the synthetic (physics-grounded) dataset

In [ ]:
from src.machine_learning.synthetic_data import generate_synthetic_roughness_degradation

df = generate_synthetic_roughness_degradation(
    diameter_m=0.0508, flow_rate_m3s=0.0005, length_m=100.0, days=730,
)
df.head()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['day'], y=df['roughness_m']*1e6, mode='markers',
                          marker=dict(size=3, opacity=0.5), name='Synthetic roughness'))
fig.update_layout(title='Synthetic Roughness Degradation Over Time',
                   xaxis_title='Day', yaxis_title='Roughness (micrometers)',
                   template='plotly_white')
fig.show()

### Fit a Random Forest regressor and evaluate on held-out data

In [ ]:
from src.machine_learning.degradation_model import fit_degradation_model, predict_roughness, predict_maintenance_threshold_day

result = fit_degradation_model(df['day'], df['roughness_m'])
print(f'Train R2: {result.train_r2:.4f}')
print(f'Test R2:  {result.test_r2:.4f}')
print(f'Test MAE: {result.test_mae:.3e} m')

In [ ]:
days_smooth = np.arange(0, 730, 5)
predicted = predict_roughness(result, days_smooth)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['day'], y=df['roughness_m']*1e6, mode='markers',
                          marker=dict(size=3, opacity=0.3), name='Synthetic data'))
fig.add_trace(go.Scatter(x=days_smooth, y=predicted*1e6, mode='lines',
                          line=dict(color='red', width=3), name='Model prediction'))
fig.update_layout(title='Random Forest Fit to Synthetic Degradation Data',
                   xaxis_title='Day', yaxis_title='Roughness (micrometers)',
                   template='plotly_white')
fig.show()

### When should maintenance be scheduled?

Find the day the model predicts roughness will reach 1.5x its initial value — a simple illustrative maintenance trigger.

In [ ]:
initial_roughness = df['roughness_m'].iloc[0]
threshold_day = predict_maintenance_threshold_day(
    result, roughness_threshold_m=initial_roughness * 1.5, max_day=730,
)

print(f'Initial roughness: {initial_roughness*1e6:.3f} micrometers')
print(f'Predicted day reaching 1.5x: day {threshold_day} (~{threshold_day/365:.2f} years)' 
      if threshold_day else 'Threshold not reached within search horizon')

**Limitation**: Random Forests don't extrapolate — predictions far beyond the training data's day range will plateau rather than continue the trend. This threshold search is only reliable within the training range (here, 0-730 days).

## Part 2: Anomaly Detection (Leak / Cavitation Onset)

### Generate synthetic sensor data with injected anomalies

In [ ]:
from src.machine_learning.synthetic_data import generate_synthetic_sensor_data_with_anomalies

sensor_df = generate_synthetic_sensor_data_with_anomalies(
    base_pressure_Pa=300_000, base_flow_m3s=0.0005,
    n_samples=400, n_anomalies=20, anomaly_magnitude_fraction=0.3,
)
sensor_df.head()

In [ ]:
fig = go.Figure()
normal = sensor_df[~sensor_df['is_true_anomaly']]
anomalies = sensor_df[sensor_df['is_true_anomaly']]
fig.add_trace(go.Scatter(x=normal['sample'], y=normal['pressure_Pa'], mode='markers',
                          marker=dict(size=4, color='steelblue'), name='Normal'))
fig.add_trace(go.Scatter(x=anomalies['sample'], y=anomalies['pressure_Pa'], mode='markers',
                          marker=dict(size=8, color='red', symbol='x'), name='True anomaly'))
fig.update_layout(title='Synthetic Pressure Readings (ground truth shown)',
                   xaxis_title='Sample', yaxis_title='Pressure (Pa)', template='plotly_white')
fig.show()

### Approach A: SPC control chart (transparent, no training needed)

In [ ]:
from src.machine_learning.anomaly_detection import (
    compute_spc_control_limits, flag_spc_anomalies, evaluate_detector_against_ground_truth,
)

# Establish baseline from the first 100 samples' normal (non-anomalous) readings
baseline_mask = ~sensor_df['is_true_anomaly'].iloc[:100]
baseline = sensor_df['pressure_Pa'].iloc[:100][baseline_mask]
limits = compute_spc_control_limits(baseline)
print(f'Center: {limits.center_line:.1f} Pa, UCL: {limits.upper_control_limit:.1f}, '
      f'LCL: {limits.lower_control_limit:.1f}')

spc_flags = flag_spc_anomalies(sensor_df['pressure_Pa'], limits)
spc_metrics = evaluate_detector_against_ground_truth(spc_flags, sensor_df['is_true_anomaly'])
print('SPC chart performance on this synthetic benchmark:', spc_metrics)

### Approach B: Isolation Forest (ML, multivariate)

In [ ]:
from src.machine_learning.anomaly_detection import fit_isolation_forest_detector, detect_anomalies

features = sensor_df[['pressure_Pa', 'flow_m3s']].values
model = fit_isolation_forest_detector(features, contamination=0.05)
iso_result = detect_anomalies(model, features)

iso_metrics = evaluate_detector_against_ground_truth(iso_result.anomaly_flags, sensor_df['is_true_anomaly'])
print('Isolation Forest performance on this synthetic benchmark:', iso_metrics)

## Takeaways

- **SPC control charts** need no training, are fully transparent (anyone can audit the UCL/LCL math), and integrate naturally with this project's existing Lean/Poka-Yoke approach — prefer these when interpretability matters or historical data is limited.
- **Isolation Forest** can incorporate multiple sensor signals at once (here, pressure *and* flow jointly) and may catch subtler multivariate patterns SPC would miss, at the cost of being a black box.
- Both numbers above are evaluated against *known* injected anomalies — a luxury real deployments don't have. Precision/recall here describe performance on this synthetic benchmark only, not a guarantee of real-world performance.